In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
df=spark.read.format('delta')\
    .load('abfss://bronze@adventwork.dfs.core.windows.net/products')

### **Deleting _rescued_data column**

In [0]:
df=df.drop("_rescued_data")

### **Schema**

In [0]:
df.printSchema()

root
 |-- ProductKey: string (nullable = true)
 |-- ProductSubcategoryKey: string (nullable = true)
 |-- ProductSKU: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- ModelName: string (nullable = true)
 |-- ProductDescription: string (nullable = true)
 |-- ProductColor: string (nullable = true)
 |-- ProductSize: string (nullable = true)
 |-- ProductStyle: string (nullable = true)
 |-- ProductCost: string (nullable = true)
 |-- ProductPrice: string (nullable = true)



### **Fixing data types**

In [0]:
df=df.withColumn('ProductKey',col('ProductKey').cast('int'))

In [0]:
df=df.withColumn('ProductSubcategoryKey',col('ProductSubcategoryKey').cast('int'))

In [0]:
df=df.withColumn('ProductSubcategoryKey',col('ProductSubcategoryKey').cast('int'))

In [0]:
df=df.withColumn('ProductCost',col('ProductSubcategoryKey').cast('decimal(18,2)'))


In [0]:
df=df.withColumn('ProductPrice',col('ProductPrice').cast('decimal(18,2)'))


### **Checking SubcategoryKey column's correctness**

In [0]:
df_prd_subcategory = spark.sql('''
                               select * from adventure_works.silver.productsubcategories
                               ''')

In [0]:
df=df.filter(col('ProductSubcategoryKey') <= (df_prd_subcategory.select(max('ProductSubcategoryKey')).collect()[0][0]))

### **Fixing string column's data quality**

In [0]:
df=df.withColumn('ProductSKU',trim(col('ProductSKU')))\
    .withColumn('ProductName',trim(col('ProductName')))\
    .withColumn('ModelName',trim(col('ModelName')))\
    .withColumn('ProductDescription',trim(col('ProductDescription')))\
    .withColumn('ProductColor',trim(col('ProductColor')))\
    .withColumn('ProductSize',trim(col('ProductSize')))\
    .withColumn('ProductStyle',trim(col('ProductStyle')))\

        
     

### **Deleting Duplicates**

In [0]:
df=df.dropDuplicates(['ProductKey'])

In [0]:
df=df.dropDuplicates(['ProductSKU'])

### **Fixing Business Logic**

In [0]:
df=df.filter(col("ProductCost") <= col("ProductPrice"))

### **Data Writing**

In [0]:
if spark.catalog.tableExists('adventure_works.silver.products'):
    df_silver_products = spark.read.table('adventure_works.silver.products')
    df=df.join(df_silver_products,on = ['ProductKey'],how='left_anti')

In [0]:
df.write.format('delta').mode('append')\
    .save('abfss://silver@adventwork.dfs.core.windows.net/products')

In [0]:
%sql
create table if not exists adventure_works.silver.products
using delta
location 'abfss://silver@adventwork.dfs.core.windows.net/products'

In [0]:
df.display()

ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice


In [0]:
%sql
select * from adventure_works.silver.products

ProductKey,ProductSubcategoryKey,ProductSKU,ProductName,ModelName,ProductDescription,ProductColor,ProductSize,ProductStyle,ProductCost,ProductPrice
452,33,LT-H903,Headlights - Weatherproof,Headlights - Weatherproof,Rugged weatherproof headlight.,NA,0,0,33.00,44.99
338,2,BK-R50B-44,"Road-650 Black, 44",Road-650,"Value-priced bike with many features of our top-of-the-line models. Has the same light, stiff frame, and the quick acceleration we're famous for.",Black,44,U,2.00,699.10
334,2,BK-R50B-60,"Road-650 Black, 60",Road-650,"Value-priced bike with many features of our top-of-the-line models. Has the same light, stiff frame, and the quick acceleration we're famous for.",Black,60,U,2.00,699.10
576,3,BK-T79U-60,"Touring-1000 Blue, 60",Touring-1000,Travel in style and comfort. Designed for maximum comfort and safety. Wide gear range takes on all hills. High-tech aluminum alloy construction provides durability without added weight.,Blue,60,U,3.00,2384.07
505,16,FR-T67U-62,"LL Touring Frame - Blue, 62",LL Touring Frame,Lightweight butted aluminum frame provides a more upright riding position for a trip around town. Our ground-breaking design provides optimum comfort.,Blue,62,U,16.00,333.42
571,3,BK-T18Y-58,"Touring-3000 Yellow, 58",Touring-3000,"All-occasion value bike with our basic comfort and safety features. Offers wider, more stable tires for a ride around town or weekend trip.",Yellow,58,U,3.00,742.35
508,16,FR-T67Y-54,"LL Touring Frame - Yellow, 54",LL Touring Frame,Lightweight butted aluminum frame provides a more upright riding position for a trip around town. Our ground-breaking design provides optimum comfort.,Yellow,54,U,16.00,333.42
522,15,SE-T762,ML Touring Seat/Saddle,ML Touring Seat/Saddle,New design relieves pressure for long rides.,NA,0,0,15.00,39.14
453,22,SH-M897-M,"Men's Sports Shorts, M",Men's Sports Shorts,Men's 8-panel racing shorts - lycra with an elastic waistband and leg grippers.,Black,M,M,22.00,59.99
247,14,FR-R92R-52,"HL Road Frame - Red, 52",HL Road Frame,Our lightest and best quality aluminum frame made from the newest alloy; it is welded and heat-treated for strength. Our innovative design results in maximum comfort and performance.,Red,52,U,14.00,1263.46
